# Automated **Gold**-Dialog Check

In [ ]:
# Mount Google Drive and set paths

from google.colab import drive
import os
import json
import pandas as pd
import re

# mount Drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
# base directory
SAVE_DIR = "/content/drive/MyDrive/Masterarbeit/Dialoge_Final_Gold"

# Paths to JSON files (GOOD & BAD dialogs)
good_path = os.path.join(SAVE_DIR, "all_gold_dialogs_generated.json")
bad_path = os.path.join(SAVE_DIR, "all_bad_gold_dialogs_generated.json")

print("GOOD JSON:", good_path)
print("BAD JSON :", bad_path)

# Check availability
assert os.path.exists(good_path), f"GOOD JSON not found at {good_path}"
assert os.path.exists(bad_path), f"BAD JSON not found at {bad_path}"

GOOD JSON: /content/drive/MyDrive/Masterarbeit/Dialoge_Final_Gold/all_gold_dialogs_generated.json
BAD JSON : /content/drive/MyDrive/Masterarbeit/Dialoge_Final_Gold/all_bad_gold_dialogs_generated.json


In [ ]:
# load JSONs with good and bad dialogs

def load_good_bad_dialogs(good_path: str, bad_path: str) -> pd.DataFrame:
  """
  Loads GOOD and BAD dialogues from the two JSON files and
  returns a joint DataFrame.
  """
  with open(good_path, "r", encoding="utf-8") as f:
    good_data = json.load(f)
  with open(bad_path, "r", encoding="utf-8") as f:
    bad_data = json.load(f)

  records = []

  # GOOD
  for entry in good_data:
      records.append({
          "dialog_id": entry.get("dialog_id", ""),
          "condition": "good",
          "recipe": entry.get("recipe", ""),
          "discussion": entry.get("discussion", "")
          })

    # BAD
  for entry in bad_data:
      records.append({
          "dialog_id": entry.get("dialog_id", ""),
          "condition": "bad",
          "recipe": entry.get("recipe", ""),
          "discussion": entry.get("discussion_bad", "")
          })

  return pd.DataFrame(records)

df_all = load_good_bad_dialogs(good_path, bad_path)
print(f"Loaded {len(df_all)} dialogs (GOOD + BAD)")
display(df_all.head())

Loaded 12 dialogs (GOOD + BAD)


,dialog_id,condition,recipe,discussion
0,R000_G,good,Chicken and Cauliflower Stir-Fry,Fat: The fat content in this recipe seems rela...
1,R001_G,good,Chicken Caesar Pasta Salad,Fat: The fat content seems moderate in this re...
2,R002_G,good,Classic Waldorf Salad,Fat: The salad's fat comes mainly from walnuts...
3,R003_G,good,Coconut Chicken Curry Stew,Fat: The stew's fat content seems relatively l...
4,R004_G,good,Italian Beef Sandwiches,Fat: Italian Beef Sandwiches have relatively l...


In [ ]:
def analyze_dialog_structure(row):
    """
    Checks basic structural properties of a dialog:
    - Number of lines
    - Fat/Carb alternation
    - Other roles
    - Length, number of questions
    - Error markers, user/system tokens
    """
    text = row["discussion"] or ""
    # Lines without blank lines
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    num_lines = len(lines)
    # Role = everything before the first ":" (or "")
    starts = [l.split(":", 1)[0] if ":" in l else "" for l in lines]

    # Alternation-Check: Fat, Carb, Fat, Carb, ...
    alt_ok = True
    if num_lines >= 2:
        for i, role in enumerate(starts):
            expected = "Fat" if i % 2 == 0 else "Carb"
            if role != expected:
                alt_ok = False
                break
    else:
        alt_ok = False

    # Number of turns per role
    fat_turns = sum(1 for s in starts if s == "Fat")
    carb_turns = sum(1 for s in starts if s == "Carb")

    # other roles (e.g. "User", "System")
    other_roles = [s for s in starts if s not in ("Fat", "Carb", "")]

    # Text statistics
    text_len = len(text)
    num_q = text.count("?")
    num_error = text.count("[ERROR")

    return {
        "num_lines": num_lines,
        "alt_ok": alt_ok,
        "fat_turns": fat_turns,
        "carb_turns": carb_turns,
        "other_roles": ",".join(sorted(set(other_roles))) if other_roles else "",
        "text_len": text_len,
        "num_questions": num_q,
        "has_error_marker": num_error > 0,
        "has_user_token": ("User:" in text) or ("System:" in text)
    }


qc_records = []
for i, row in df_all.iterrows():
    stats = analyze_dialog_structure(row)
    stats.update({
        "dialog_id": row["dialog_id"],
        "condition": row["condition"],
        "recipe": row["recipe"]
    })
    qc_records.append(stats)

df_qc = pd.DataFrame(qc_records)
print("QC overview (first lines):")
display(df_qc.head())

QC overview (first lines):


,num_lines,alt_ok,fat_turns,carb_turns,other_roles,text_len,num_questions,has_error_marker,has_user_token,dialog_id,condition,recipe
0,20,True,10,10,,1288,0,False,False,R000_G,good,Chicken and Cauliflower Stir-Fry
1,20,True,10,10,,1793,1,False,False,R001_G,good,Chicken Caesar Pasta Salad
2,20,True,10,10,,1739,1,False,False,R002_G,good,Classic Waldorf Salad
3,20,True,10,10,,1420,1,False,False,R003_G,good,Coconut Chicken Curry Stew
4,20,True,10,10,,1442,2,False,False,R004_G,good,Italian Beef Sandwiches


In [ ]:
df_qc.to_csv('/content/drive/MyDrive/Masterarbeit/Dialoge_Final_Gold/automated_structure_check_gold.csv', index=False)

In [ ]:
# Filtering problem cases

# Criteria for “conspicuous”
problematic = df_qc[
    (df_qc["num_lines"] != 20) |      # not exactly 20 turns
    (df_qc["alt_ok"] == False) |      # wrong order
    (df_qc["has_error_marker"]) |     # Error-Markers
    (df_qc["has_user_token"]) |       # User/System-Token
    (df_qc["text_len"] < 300)         # very short dialogs
]

print(f"Conspicuous dialogues: {len(problematic)}")
display(problematic.sort_values(["condition", "dialog_id"]).head(50))

# Save as CSV for later review
problematic.to_csv("dialog_qc_flags.csv", index=False)
print("Saved QC flags to dialog_qc_flags.csv")

Conspicuous dialogues: 0


,num_lines,alt_ok,fat_turns,carb_turns,other_roles,text_len,num_questions,has_error_marker,has_user_token,dialog_id,condition,recipe


Saved QC flags to dialog_qc_flags.csv
